# Paper 1 — Part 2 experiments (KAGGLE)   notebook version P1-part2 1.0

**Inputs (Add Input):** (1) dataset **citrus-data**; (2) notebook output **notebook4b9b056063**.
**Accelerator:** GPU T4 x1. **Run:** Save Version → Save & Run All (Commit). Expected time about 6–8 hours.

What it does (no earlier result is changed):
* Cell 6 — quick tasks: EXIF timestamps (M14), fruit-only source probe (3.1), image sheets for the cross-class and detector-validation pairs (C4, C5)
* Cell 7 — C3 deduplicated arm (one image per group, ResNet50 + MobileNetV2)
* Cell 8 — M1/M2 leakage simulation with 5 seeds, protocols A, A′ and B
* Cell 9 — 2.10 grouped CV within each dataset
* Cell 10 — M3 random split for the other five architectures
* Cell 11 — M4 grouped CV with two more fold assignments, all seven architectures
* Cell 12 — pack results into **p1_part2_results.zip** (download only this file)

Every training cell is resumable: if the run stops, attach this notebook's own output as an extra input in a NEW notebook and run again; finished parts are skipped.

In [ ]:
#@title Cell 1 — Settings   [notebook version P1-part2 1.0]
SEED = 42
INPUT_ROOT = "/kaggle/input"
OUT = "/kaggle/working/citrus_p1part2_results"
N_FOLDS = 5
HEAD_EP, FT_EP, FT_LAYERS, BATCH = 10, 8, 30, 32      # identical to the main benchmark
ALL_MODELS = ["ResNet50", "MobileNetV2", "EfficientNetB0", "DenseNet121", "InceptionV3", "EfficientNetV2S", "ConvNeXtTiny"]
TWO = ["ResNet50", "MobileNetV2"]
LEAK_SEEDS = [0, 1, 2, 3, 4]
REPEAT_FOLD_SEEDS = [0, 1]
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONHASHSEED"] = str(SEED)
print("notebook version P1-part2 1.0")
BR_P_KEEP, BR_P_COLOUR = 0.2, 0.4   # only referenced by an unused helper in Cell 5


In [ ]:
#@title Cell 2 — Install, locate inputs, restore fold assignment (and resume an earlier run of this notebook)
!pip -q install statsmodels scikit-image imagehash
import glob, zipfile, json, math, random, time, shutil
import numpy as np, pandas as pd
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
CLS_CHECK = ["Black_Spot", "Canker", "Greening", "Healthy", "Melanose"]

def is_data_root(d):
    return (os.path.isdir(f"{d}/train") and os.path.isdir(f"{d}/test")
            and sum(os.path.isdir(f"{d}/train/{c}") for c in CLS_CHECK) >= 3)
DATA = None
for root, dirs, files in os.walk(INPUT_ROOT):
    if is_data_root(root) and "SMOKE" not in root:
        DATA = root; break
if DATA is None:
    zips = glob.glob(f"{INPUT_ROOT}/**/split_dataset*.zip", recursive=True)
    assert zips, "split_dataset not found. Add the 'citrus-data' dataset via Add Input."
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/temp/data")
    for root, dirs, files in os.walk("/kaggle/temp/data"):
        if is_data_root(root):
            DATA = root; break
assert DATA
print("Images from:", DATA)
SUBS = ["tables", "oof_dedup", "oof_leak", "oof_within", "oof_random", "oof_repeat", "sheets"]
for sub in SUBS:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
# resume: if the output of an earlier (stopped) run of THIS notebook is attached as input, restore its logs and predictions
for root, dirs, files in os.walk(INPUT_ROOT):
    if os.path.basename(root) == "citrus_p1part2_results" and os.path.isdir(f"{root}/tables"):
        for sub in SUBS:
            if os.path.isdir(f"{root}/{sub}"):
                shutil.copytree(f"{root}/{sub}", f"{OUT}/{sub}", dirs_exist_ok=True)
        print("Resumed from earlier output of this notebook:", root)

# earlier results (only tables and out-of-fold predictions are copied; weights are not needed)
prev_dirs = []
for root, dirs, files in os.walk(INPUT_ROOT):
    if "SMOKE" in root or "citrus_v3" in root or "citrus_v4" in root or "citrus_p1part2" in root:
        continue
    if os.path.isdir(f"{root}/tables") and os.path.isdir(f"{root}/oof"):
        prev_dirs.append(root)
FOLD_SRC = None
for p in prev_dirs:
    f = f"{p}/tables/images_groups_folds.csv"
    if FOLD_SRC is None and os.path.exists(f):
        FOLD_SRC = f
    for t in ["cross_class_near_duplicates.csv", "model_summary.csv"]:
        if os.path.exists(f"{p}/tables/{t}") and not os.path.exists(f"{OUT}/tables/prev_{t}"):
            shutil.copy(f"{p}/tables/{t}", f"{OUT}/tables/prev_{t}")
assert FOLD_SRC, "images_groups_folds.csv not found: add the output of notebook4b9b056063 (or notebook152fe2c903) via Add Input."
shutil.copy(FOLD_SRC, f"{OUT}/tables/images_groups_folds.csv")
print("Earlier results found in:", prev_dirs)
print("Cross-class pair list found:", os.path.exists(f"{OUT}/tables/prev_cross_class_near_duplicates.csv"))

import tensorflow as tf
import keras
from keras import layers
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
print("TF", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title Cell 3 — Pool all 1,834 images and tag source / organ (unchanged)
CLASSES = ["Black_Spot", "Canker", "Greening", "Healthy", "Melanose"]
NC = len(CLASSES)
rows = []
for split in ["train", "val", "test"]:
    for c in CLASSES:
        for f in sorted(glob.glob(f"{DATA}/{split}/{c}/*")):
            b = os.path.basename(f)
            src = "Rauf" if b.startswith("citrus_") else ("Silva" if b.startswith("archive1_") else "Other")
            organ = "leaf" if (src == "Rauf" and "Leaves" in b) else "fruit"
            rows.append(dict(path=f, file=b, cls=c, y=CLASSES.index(c), source=src, organ=organ, old_split=split))
df = pd.DataFrame(rows)
N = len(df)
print(N, "images")
display(pd.crosstab([df.source, df.organ], df.cls, margins=True))

def load(p, size=224):
    return np.asarray(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)
X = np.stack([load(p) for p in df.path])
def black_corners(x):
    return sum(x[r:r + 12, c:c + 12].mean() < 20 for r, c in [(0, 0), (0, 212), (212, 0), (212, 212)])
df["pre_rotated"] = [black_corners(x) >= 2 for x in X]
print("pre-rotated images:", int(df.pre_rotated.sum()))
Y = df.y.values.astype("int64")


In [ ]:
#@title Cell 4 — Re-use the grouped, stratified 5-fold split of the main benchmark
from sklearn.model_selection import StratifiedGroupKFold
FOLD_CSV = f"{OUT}/tables/images_groups_folds.csv"
reused = False
if os.path.exists(FOLD_CSV):
    prev = pd.read_csv(FOLD_CSV)
    kp = prev.old_split + "|" + prev.cls + "|" + prev.file
    kd = df.old_split + "|" + df.cls + "|" + df.file
    if len(prev) == N and kd.isin(set(kp)).all():
        df["group"] = kd.map(dict(zip(kp, prev.group))).values
        df["fold"] = kd.map(dict(zip(kp, prev.fold))).values.astype(int)
        reused = True
        print("Re-using the group and fold assignment saved by the earlier run (finished folds stay valid).")
assert reused, "The fold file of the main benchmark was not matched; do not continue."
G = df.group.values
if not reused:
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    df["fold"] = -1
    for k, (_, te) in enumerate(sgkf.split(np.zeros(N), Y, G)):
        df.loc[te, "fold"] = k
display(pd.crosstab(df.fold, df.cls))

def split_fold(k):
    te = np.where(df.fold.values == k)[0]
    rest = np.where(df.fold.values != k)[0]
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED + k)
    tr_i, va_i = next(inner.split(np.zeros(len(rest)), Y[rest], G[rest]))
    return rest[tr_i], rest[va_i], te
for k in range(N_FOLDS):
    tr, va, te = split_fold(k)
    assert not (set(G[tr]) & set(G[te])) and not (set(G[va]) & set(G[te])) and not (set(G[tr]) & set(G[va]))
print("Check passed: no near-duplicate group is shared between train, validation and test in any fold.")
df.drop(columns=["path"]).to_csv(f"{OUT}/tables/images_groups_folds.csv", index=False)


In [ ]:
#@title Cell 5 — Models, data pipeline, training helpers, object masks and background randomisation
APPS = {
    "ResNet50": (keras.applications.ResNet50, keras.applications.resnet.preprocess_input),
    "MobileNetV2": (keras.applications.MobileNetV2, keras.applications.mobilenet_v2.preprocess_input),
    "EfficientNetB0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
    "DenseNet121": (keras.applications.DenseNet121, keras.applications.densenet.preprocess_input),
    "InceptionV3": (keras.applications.InceptionV3, keras.applications.inception_v3.preprocess_input),
    "EfficientNetV2S": (keras.applications.EfficientNetV2S, keras.applications.efficientnet_v2.preprocess_input),
    "ConvNeXtTiny": (keras.applications.ConvNeXtTiny, keras.applications.convnext.preprocess_input),
}
AUTOTUNE = tf.data.AUTOTUNE
AUG = keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.1), layers.RandomZoom(0.2),
                        layers.RandomContrast(0.2), layers.RandomBrightness(0.2, value_range=(0, 255))])

def build_model(name, n_out):
    ctor, _ = APPS[name]
    inp = keras.Input((224, 224, 3))
    base = ctor(include_top=False, weights="imagenet", input_tensor=inp)
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_out, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out), base

def _batch_index(ya, seed, n_batches):
    """Square-root class-balanced sampling with replacement (same rule as before), as a fixed index table."""
    rng = np.random.default_rng(seed)
    cls = np.unique(ya); idx = [np.where(ya == c)[0] for c in cls]
    w = np.sqrt([len(i) for i in idx]); w = w / w.sum()
    out = np.empty((n_batches, BATCH), dtype="int64")
    for b in range(n_batches):
        cs = rng.choice(len(cls), size=BATCH, p=w)
        out[b] = [rng.choice(idx[c]) for c in cs]
    return out

def train_ds(Xa, ya, prep):
    with tf.device("/CPU:0"):
        Xt = tf.convert_to_tensor(Xa); yt = tf.convert_to_tensor(np.asarray(ya, dtype="int64"))
    steps = max(1, math.ceil(len(ya) / BATCH))
    ds = tf.data.Dataset.from_tensor_slices(_batch_index(np.asarray(ya), SEED, steps * max(HEAD_EP, FT_EP))).repeat()
    ds = ds.map(lambda i: (tf.gather(Xt, i), tf.gather(yt, i)))
    return ds.map(lambda x, y: (prep(AUG(tf.cast(x, tf.float32), training=True)), y), num_parallel_calls=AUTOTUNE).prefetch(2)

def eval_ds(Xa, prep, ya=None, fn=None):
    if fn is not None:
        Xa = np.stack([fn(x) for x in Xa])
    if ya is None:
        return tf.data.Dataset.from_tensor_slices(Xa).batch(BATCH).map(lambda x: prep(tf.cast(x, tf.float32))).prefetch(2)
    ds = tf.data.Dataset.from_tensor_slices((Xa, np.asarray(ya, dtype="int64"))).batch(BATCH)
    return ds.map(lambda x, y: (prep(tf.cast(x, tf.float32)), y)).prefetch(2)

def ram_gb():
    try:
        import psutil; return psutil.Process().memory_info().rss / 1e9
    except Exception:
        return float("nan")
def ram_limit_gb():
    try:
        import psutil; return 0.75 * psutil.virtual_memory().total / 1e9
    except Exception:
        return 1e9

def free_memory(*objs):
    import gc
    for o in objs: del o
    keras.backend.clear_session(); gc.collect(); gc.collect()

def train_one(name, Xtr, ytr, Xva, yva, n_out):
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    model, base = build_model(name, n_out)
    prep = APPS[name][1]
    tr, va = train_ds(Xtr, ytr, prep), eval_ds(Xva, prep, yva)
    steps = max(1, math.ceil(len(ytr) / BATCH))
    loss = "sparse_categorical_crossentropy"
    for l in base.layers: l.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=HEAD_EP, steps_per_epoch=steps, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
    for l in base.layers[-FT_LAYERS:]:
        if not isinstance(l, layers.BatchNormalization): l.trainable = True
    model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=loss, metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=FT_EP, steps_per_epoch=steps, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
    return model, base

def predict_u8(model, name, Xa, fn=None):
    return model.predict(eval_ds(Xa, APPS[name][1], fn=fn), verbose=0)

def centre_only(x, keep=0.7):
    h = x.shape[0]; m = int(h * (1 - keep) / 2)
    out = np.full_like(x, 124); out[m:h - m, m:h - m] = x[m:h - m, m:h - m]; return out
def corners_only(x, s=56):
    h = x.shape[0]; out = np.full_like(x, 124)
    for r, c in [(0, 0), (0, h - s), (h - s, 0), (h - s, h - s)]:
        out[r:r + s, c:c + s] = x[r:r + s, c:c + s]
    return out

from scipy import ndimage as ndi
from skimage import color, filters, measure
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
def object_mask(x):
    """Leaf/fruit mask on plain backgrounds. Black padding or black backdrops are treated as background.
    Returns None when the mask is unreliable (cluttered field scenes, fragmented or tiny/huge masks)."""
    fill = x.max(axis=-1) < 25
    lab = color.rgb2lab(x)
    ring = np.zeros(x.shape[:2], bool); ring[:8] = ring[-8:] = True; ring[:, :8] = ring[:, -8:] = True
    bsel = ring & ~fill
    if bsel.sum() < 0.25 * ring.sum():          # border is mostly black: dark backdrop or padding
        m = ndi.binary_opening(x.max(axis=-1) >= 40, structure=np.ones((5, 5)))
    else:
        d = np.linalg.norm(lab - np.median(lab[bsel], 0), axis=-1); d[fill] = 0
        if (~fill).sum() < 100 or np.ptp(d[~fill]) == 0: return None
        m = ndi.binary_opening(d > filters.threshold_otsu(d[~fill]), structure=np.ones((5, 5))) & ~fill
    lbl = measure.label(m)
    if lbl.max() == 0: return None
    h = x.shape[0]; c0, c1 = h // 4, 3 * h // 4
    centre = np.bincount(lbl[c0:c1, c0:c1].ravel(), minlength=lbl.max() + 1)[1:]
    if centre.max() == 0: return None
    m = ndi.binary_fill_holes(lbl == (int(np.argmax(centre)) + 1))
    if not (0.05 < m.mean() < 0.95): return None
    if measure.regionprops(m.astype(np.uint8))[0].solidity < 0.6: return None   # fragmented -> unreliable
    return m

def randomise_background(x, m):
    """x: float32 batch (0-255), m: float32 masks (1 = leaf/fruit). Keeps the object, replaces the background
    by a random colour or a smooth random texture; images without a reliable mask have m = 1 everywhere."""
    b = tf.shape(x)[0]
    colour = tf.random.uniform((b, 1, 1, 3), 0.0, 255.0) * tf.ones_like(x)
    tex = tf.image.resize(tf.random.uniform((b, 28, 28, 3), 0.0, 255.0), (224, 224), method="bicubic")
    tex = tf.clip_by_value(tf.nn.avg_pool2d(tex, 9, 1, "SAME"), 0.0, 255.0)
    u = tf.random.uniform((b, 1, 1, 1))
    bg = tf.where(u < BR_P_KEEP + BR_P_COLOUR, colour, tex)
    return tf.where(u < BR_P_KEEP, x, m * x + (1.0 - m) * bg)

def train_ds2(Xa, ya, prep, seed, Ma=None):
    with tf.device("/CPU:0"):
        Xt = tf.convert_to_tensor(Xa); yt = tf.convert_to_tensor(np.asarray(ya, dtype="int64"))
        Mt = None if Ma is None else tf.convert_to_tensor(Ma[..., None])
    steps = max(1, math.ceil(len(ya) / BATCH))
    ds = tf.data.Dataset.from_tensor_slices(_batch_index(np.asarray(ya), seed, steps * max(HEAD_EP, FT_EP))).repeat()
    if Mt is None:
        ds = ds.map(lambda i: (tf.gather(Xt, i), tf.gather(yt, i)))
        return ds.map(lambda x, y: (prep(AUG(tf.cast(x, tf.float32), training=True)), y), num_parallel_calls=AUTOTUNE).prefetch(2)
    ds = ds.map(lambda i: (tf.gather(Xt, i), tf.gather(Mt, i), tf.gather(yt, i)))
    return ds.map(lambda x, m, y: (prep(AUG(randomise_background(tf.cast(x, tf.float32), tf.cast(m, tf.float32)), training=True)), y),
                  num_parallel_calls=AUTOTUNE).prefetch(2)

def train_two(name, Xtr, ytr, Xva, yva, n_out, Mtr=None, seed=SEED):
    """Same two-stage recipe as the main benchmark; Mtr=None gives the baseline, masks give background randomisation."""
    keras.backend.clear_session(); keras.utils.set_random_seed(seed)
    model, base = build_model(name, n_out)
    prep = APPS[name][1]
    tr, va = train_ds2(Xtr, ytr, prep, seed, Mtr), eval_ds(Xva, prep, yva)
    steps = max(1, math.ceil(len(ytr) / BATCH))
    loss = "sparse_categorical_crossentropy"
    for l in base.layers: l.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=HEAD_EP, steps_per_epoch=steps, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
    for l in base.layers[-FT_LAYERS:]:
        if not isinstance(l, layers.BatchNormalization): l.trainable = True
    model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=loss, metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=FT_EP, steps_per_epoch=steps, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
    return model, base

def mf1(y, p, labels=None):
    return f1_score(y, p, average="macro", labels=labels if labels is not None else np.unique(y), zero_division=0)
def wilson(c, n):
    lo, hi = proportion_confint(c, n, method="wilson"); return f"{lo:.3f}-{hi:.3f}"
def append_log(path, row):
    L = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    pd.concat([L, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)


In [ ]:
C5_JSON = r"""[["P001", "archive1_train_grenning__g (171).jpg", "archive1_train_grenning__g (152).jpg"], ["P002", "archive1_train_canker__c (26).jpg", "archive1_train_canker__c (157).jpg"], ["P003", "archive1_train_blackspot__b (176).jpg", "archive1_train_blackspot__b (22).jpg"], ["P004", "archive1_train_grenning__g (139).jpg", "archive1_train_grenning__g (264).jpg"], ["P005", "archive1_train_canker__c (145).jpg", "archive1_train_canker__c (160).jpg"], ["P006", "archive1_train_grenning__g (19).jpg", "archive1_train_grenning__g (253).jpg"], ["P007", "archive1_train_canker__c (68).jpg", "archive1_train_canker__c (65).jpg"], ["P008", "archive1_train_grenning__g (276).jpg", "archive1_train_grenning__g (36).jpg"], ["P009", "archive1_train_grenning__g (98).jpg", "archive1_train_grenning__g (92).jpg"], ["P010", "archive1_train_grenning__g (131).jpg", "archive1_train_grenning__g (307).jpg"], ["P011", "citrus_Citrus_Leaves_Blackspot__b (58).png", "citrus_Citrus_Leaves_Blackspot__b (74).png"], ["P012", "archive1_train_grenning__g (249).jpg", "archive1_train_grenning__g (208).jpg"], ["P013", "archive1_train_grenning__g (231).jpg", "archive1_train_grenning__g (58).jpg"], ["P014", "archive1_train_grenning__g (150).jpg", "archive1_train_grenning__g (60).jpg"], ["P015", "archive1_train_blackspot__b (120).jpg", "archive1_train_blackspot__b (136).jpg"], ["P016", "archive1_train_grenning__g (125).jpg", "archive1_train_grenning__g (129).jpg"], ["P017", "archive1_train_blackspot__b (38).jpg", "archive1_train_blackspot__b (43).jpg"], ["P018", "archive1_train_grenning__g (122).jpg", "archive1_train_grenning__g (186).jpg"], ["P019", "archive1_train_grenning__g (142).jpg", "archive1_train_grenning__g (157).jpg"], ["P020", "archive1_train_blackspot__b (46).jpg", "archive1_train_blackspot__b (106).jpg"], ["P021", "archive1_train_grenning__g (133).jpg", "archive1_train_grenning__g (270).jpg"], ["P022", "archive1_train_grenning__g (186).jpg", "archive1_train_grenning__g (175).jpg"], ["P023", "archive1_train_canker__c (49).jpg", "archive1_train_canker__c (73).jpg"], ["P024", "archive1_test_grenning__g (10).jpg", "archive1_train_grenning__g (6).jpg"], ["P025", "archive1_train_canker__c (57).jpg", "archive1_train_canker__c (157).jpg"], ["P026", "archive1_train_blackspot__b (139).jpg", "archive1_train_blackspot__b (13).jpg"], ["P027", "archive1_train_blackspot__b (156).jpg", "archive1_train_blackspot__b (135).jpg"], ["P028", "archive1_train_canker__c (28).jpg", "archive1_train_canker__c (29).jpg"], ["P029", "archive1_train_grenning__g (305).jpg", "archive1_train_grenning__g (72).jpg"], ["P030", "archive1_train_canker__c (172).jpg", "archive1_train_canker__c (30).jpg"], ["P031", "archive1_train_grenning__g (192).jpg", "archive1_train_grenning__g (119).jpg"], ["P032", "archive1_train_grenning__g (41).jpg", "archive1_train_grenning__g (60).jpg"], ["P033", "archive1_train_canker__c (27).jpg", "archive1_train_canker__c (130).jpg"], ["P034", "archive1_train_grenning__g (127).jpg", "archive1_train_grenning__g (322).jpg"], ["P035", "archive1_train_grenning__g (155).jpg", "archive1_train_grenning__g (192).jpg"], ["P036", "archive1_train_grenning__g (12).jpg", "archive1_train_grenning__g (271).jpg"], ["P037", "archive1_train_grenning__g (85).jpg", "archive1_train_grenning__g (86).jpg"], ["P038", "archive1_train_grenning__g (19).jpg", "archive1_train_grenning__g (72).jpg"], ["P039", "archive1_train_grenning__g (150).jpg", "archive1_train_grenning__g (11).jpg"], ["P040", "archive1_train_blackspot__b (52).jpg", "archive1_train_blackspot__b (26).jpg"], ["P041", "archive1_train_blackspot__b (8).jpg", "archive1_train_blackspot__b (171).jpg"], ["P042", "archive1_train_blackspot__b (171).jpg", "archive1_train_blackspot__b (19).jpg"], ["P043", "archive1_train_canker__c (48).jpg", "archive1_train_canker__c (56).jpg"], ["P044", "archive1_train_grenning__g (156).jpg", "archive1_train_grenning__g (151).jpg"], ["P045", "archive1_train_grenning__g (321).jpg", "archive1_train_grenning__g (63).jpg"], ["P046", "archive1_train_blackspot__b (91).jpg", "archive1_train_blackspot__b (66).jpg"], ["P047", "archive1_train_canker__c (57).jpg", "archive1_train_canker__c (83).jpg"], ["P048", "archive1_train_grenning__g (109).jpg", "archive1_train_grenning__g (207).jpg"], ["P049", "archive1_test_blackspot__b (13).jpg", "archive1_train_blackspot__b (20).jpg"], ["P050", "archive1_test_fresh__f (11).png", "archive1_train_fresh__f (122).png"], ["P051", "archive1_train_canker__c (126).jpg", "archive1_train_canker__c (159).jpg"], ["P052", "archive1_train_grenning__g (330).jpg", "archive1_train_grenning__g (97).jpg"], ["P053", "archive1_train_grenning__g (147).jpg", "archive1_train_grenning__g (318).jpg"], ["P054", "archive1_train_fresh__f (130).png", "archive1_train_fresh__f (26).png"], ["P055", "archive1_train_grenning__g (147).jpg", "archive1_train_grenning__g (77).jpg"], ["P056", "archive1_train_grenning__g (280).jpg", "archive1_train_grenning__g (83).jpg"], ["P057", "archive1_train_grenning__g (28).jpg", "archive1_train_grenning__g (52).jpg"], ["P058", "archive1_train_canker__c (146).jpg", "archive1_train_canker__c (111).jpg"], ["P059", "archive1_train_blackspot__b (20).jpg", "archive1_train_blackspot__b (62).jpg"], ["P060", "archive1_train_blackspot__b (10).jpg", "archive1_train_blackspot__b (55).jpg"], ["P061", "archive1_train_blackspot__b (36).jpg", "archive1_train_blackspot__b (37).jpg"], ["P062", "archive1_train_blackspot__b (105).jpg", "archive1_train_blackspot__b (62).jpg"], ["P063", "archive1_train_grenning__g (7).jpg", "archive1_test_grenning__g (7).jpg"], ["P064", "archive1_test_canker__c (13).jpg", "archive1_train_canker__c (42).jpg"], ["P065", "archive1_train_blackspot__b (2).jpg", "archive1_train_blackspot__b (32).jpg"], ["P066", "archive1_train_grenning__g (286).jpg", "archive1_train_grenning__g (77).jpg"], ["P067", "archive1_test_blackspot__b (5).jpg", "archive1_train_blackspot__b (100).jpg"], ["P068", "archive1_train_blackspot__b (15).jpg", "archive1_train_blackspot__b (36).jpg"], ["P069", "archive1_test_blackspot__b (13).jpg", "archive1_train_blackspot__b (160).jpg"], ["P070", "archive1_train_blackspot__b (88).jpg", "archive1_train_blackspot__b (145).jpg"], ["P071", "archive1_train_blackspot__b (105).jpg", "archive1_train_blackspot__b (154).jpg"], ["P072", "archive1_train_blackspot__b (78).jpg", "archive1_train_blackspot__b (91).jpg"], ["P073", "archive1_train_grenning__g (112).jpg", "archive1_train_grenning__g (249).jpg"], ["P074", "archive1_train_blackspot__b (56).jpg", "archive1_train_blackspot__b (141).jpg"], ["P075", "archive1_test_grenning__g (1).jpg", "archive1_train_grenning__g (286).jpg"], ["P076", "archive1_train_grenning__g (165).jpg", "archive1_train_grenning__g (214).jpg"], ["P077", "archive1_train_grenning__g (307).jpg", "archive1_train_grenning__g (58).jpg"], ["P078", "archive1_train_grenning__g (80).jpg", "archive1_train_grenning__g (57).jpg"], ["P079", "archive1_train_canker__c (110).jpg", "archive1_train_canker__c (126).jpg"], ["P080", "archive1_train_grenning__g (79).jpg", "archive1_train_grenning__g (319).jpg"], ["P081", "archive1_test_grenning__g (5).jpg", "archive1_train_grenning__g (221).jpg"], ["P082", "archive1_test_grenning__g (2).jpg", "archive1_train_grenning__g (252).jpg"], ["P083", "archive1_train_grenning__g (136).jpg", "archive1_train_grenning__g (206).jpg"], ["P084", "archive1_train_blackspot__b (122).jpg", "archive1_train_blackspot__b (121).jpg"], ["P085", "archive1_train_grenning__g (283).jpg", "archive1_train_grenning__g (15).jpg"], ["P086", "archive1_test_blackspot__b (8).jpg", "archive1_train_blackspot__b (47).jpg"], ["P087", "archive1_train_canker__c (57).jpg", "archive1_train_canker__c (82).jpg"], ["P088", "archive1_train_grenning__g (273).jpg", "archive1_train_grenning__g (119).jpg"], ["P089", "archive1_train_grenning__g (291).jpg", "archive1_train_grenning__g (83).jpg"], ["P090", "archive1_train_grenning__g (112).jpg", "archive1_train_grenning__g (18).jpg"], ["P091", "archive1_train_blackspot__b (104).jpg", "archive1_train_blackspot__b (66).jpg"], ["P092", "archive1_train_grenning__g (346).jpg", "archive1_test_grenning__g (15).jpg"], ["P093", "archive1_test_blackspot__b (11).jpg", "archive1_train_blackspot__b (11).jpg"], ["P094", "archive1_test_grenning__g (3).jpg", "archive1_train_grenning__g (6).jpg"], ["P095", "archive1_test_grenning__g (2).jpg", "archive1_train_grenning__g (69).jpg"], ["P096", "archive1_test_grenning__g (1).jpg", "archive1_train_grenning__g (107).jpg"], ["P097", "archive1_train_grenning__g (137).jpg", "archive1_train_grenning__g (87).jpg"], ["P098", "archive1_train_canker__c (75).jpg", "archive1_train_canker__c (29).jpg"], ["P099", "archive1_train_blackspot__b (38).jpg", "archive1_train_blackspot__b (64).jpg"], ["P100", "archive1_train_blackspot__b (36).jpg", "archive1_train_blackspot__b (72).jpg"], ["P101", "archive1_test_grenning__g (9).jpg", "archive1_train_grenning__g (126).jpg"], ["P102", "archive1_train_grenning__g (261).jpg", "archive1_train_grenning__g (210).jpg"], ["P103", "archive1_train_grenning__g (133).jpg", "archive1_train_grenning__g (305).jpg"], ["P104", "archive1_train_grenning__g (108).jpg", "archive1_train_grenning__g (207).jpg"], ["P105", "archive1_train_canker__c (30).jpg", "archive1_train_canker__c (55).jpg"], ["P106", "archive1_train_blackspot__b (154).jpg", "archive1_train_blackspot__b (96).jpg"], ["P107", "archive1_test_canker__c (21).jpg", "archive1_train_canker__c (165).jpg"], ["P108", "archive1_train_grenning__g (239).jpg", "archive1_train_grenning__g (98).jpg"], ["P109", "archive1_train_grenning__g (126).jpg", "archive1_train_grenning__g (97).jpg"], ["P110", "archive1_train_grenning__g (233).jpg", "archive1_train_grenning__g (336).jpg"], ["P111", "archive1_test_blackspot__b (18).jpg", "archive1_train_blackspot__b (69).jpg"], ["P112", "archive1_train_grenning__g (107).jpg", "archive1_train_grenning__g (156).jpg"], ["P113", "archive1_train_canker__c (155).jpg", "archive1_train_canker__c (128).jpg"], ["P114", "archive1_train_blackspot__b (178).jpg", "archive1_train_blackspot__b (61).jpg"], ["P115", "archive1_train_grenning__g (127).jpg", "archive1_train_grenning__g (276).jpg"], ["P116", "archive1_train_grenning__g (39).jpg", "archive1_train_grenning__g (89).jpg"], ["P117", "archive1_train_blackspot__b (96).jpg", "archive1_train_blackspot__b (125).jpg"], ["P118", "archive1_train_grenning__g (345).jpg", "archive1_train_grenning__g (138).jpg"], ["P119", "archive1_train_grenning__g (246).jpg", "archive1_train_grenning__g (271).jpg"], ["P120", "archive1_train_blackspot__b (166).jpg", "archive1_train_blackspot__b (171).jpg"], ["P121", "archive1_train_grenning__g (243).jpg", "archive1_train_grenning__g (12).jpg"], ["P122", "archive1_train_grenning__g (100).jpg", "archive1_train_grenning__g (237).jpg"], ["P123", "archive1_train_canker__c (85).jpg", "archive1_train_canker__c (110).jpg"], ["P124", "archive1_train_blackspot__b (167).jpg", "archive1_train_blackspot__b (75).jpg"], ["P125", "archive1_train_grenning__g (205).jpg", "archive1_train_grenning__g (64).jpg"], ["P126", "archive1_train_canker__c (165).jpg", "archive1_train_canker__c (168).jpg"], ["P127", "archive1_train_grenning__g (288).jpg", "archive1_train_grenning__g (271).jpg"], ["P128", "archive1_train_blackspot__b (181).jpg", "archive1_train_blackspot__b (33).jpg"], ["P129", "archive1_train_grenning__g (263).jpg", "archive1_train_grenning__g (248).jpg"], ["P130", "archive1_train_canker__c (52).jpg", "archive1_train_canker__c (80).jpg"], ["P131", "archive1_train_canker__c (48).jpg", "archive1_train_canker__c (157).jpg"], ["P132", "archive1_train_grenning__g (116).jpg", "archive1_train_grenning__g (48).jpg"], ["P133", "archive1_train_blackspot__b (116).jpg", "archive1_train_blackspot__b (163).jpg"], ["P134", "archive1_train_canker__c (127).jpg", "archive1_train_canker__c (128).jpg"], ["P135", "archive1_train_grenning__g (261).jpg", "archive1_train_grenning__g (305).jpg"], ["P136", "archive1_train_grenning__g (171).jpg", "archive1_train_grenning__g (296).jpg"], ["P137", "archive1_train_grenning__g (29).jpg", "archive1_train_grenning__g (289).jpg"], ["P138", "archive1_train_blackspot__b (172).jpg", "archive1_train_blackspot__b (11).jpg"], ["P139", "archive1_test_blackspot__b (11).jpg", "archive1_train_blackspot__b (15).jpg"], ["P140", "archive1_train_blackspot__b (49).jpg", "archive1_train_blackspot__b (64).jpg"], ["P141", "archive1_train_grenning__g (172).jpg", "archive1_train_grenning__g (53).jpg"], ["P142", "archive1_train_canker__c (57).jpg", "archive1_train_canker__c (58).jpg"], ["P143", "archive1_train_blackspot__b (15).jpg", "archive1_train_blackspot__b (54).jpg"], ["P144", "archive1_test_blackspot__b (17).jpg", "archive1_train_blackspot__b (66).jpg"], ["P145", "archive1_test_blackspot__b (13).jpg", "archive1_train_blackspot__b (22).jpg"], ["P146", "archive1_train_grenning__g (33).jpg", "archive1_train_grenning__g (53).jpg"], ["P147", "archive1_train_grenning__g (162).jpg", "archive1_train_grenning__g (280).jpg"], ["P148", "archive1_train_canker__c (128).jpg", "archive1_train_canker__c (130).jpg"], ["P149", "archive1_train_grenning__g (77).jpg", "archive1_train_grenning__g (60).jpg"], ["P150", "archive1_train_grenning__g (155).jpg", "archive1_train_grenning__g (297).jpg"], ["P151", "archive1_train_blackspot__b (109).jpg", "archive1_train_blackspot__b (107).jpg"], ["P152", "archive1_train_blackspot__b (73).jpg", "archive1_train_blackspot__b (86).jpg"], ["P153", "citrus_Citrus_Leaves_greening__g (163).png", "citrus_Citrus_Leaves_greening__g (181).png"], ["P154", "archive1_train_grenning__g (295).jpg", "archive1_train_grenning__g (35).jpg"], ["P155", "archive1_train_grenning__g (181).jpg", "archive1_train_grenning__g (151).jpg"], ["P156", "archive1_train_grenning__g (176).jpg", "archive1_train_grenning__g (240).jpg"], ["P157", "archive1_train_blackspot__b (138).jpg", "archive1_train_blackspot__b (125).jpg"], ["P158", "archive1_train_grenning__g (150).jpg", "archive1_train_grenning__g (174).jpg"], ["P159", "archive1_train_grenning__g (162).jpg", "archive1_train_grenning__g (243).jpg"], ["P160", "archive1_train_grenning__g (33).jpg", "archive1_train_grenning__g (229).jpg"], ["P161", "archive1_train_grenning__g (309).jpg", "archive1_train_grenning__g (63).jpg"], ["P162", "archive1_train_blackspot__b (183).jpg", "archive1_train_blackspot__b (65).jpg"], ["P163", "archive1_train_grenning__g (121).jpg", "archive1_train_grenning__g (129).jpg"], ["P164", "archive1_train_grenning__g (318).jpg", "archive1_train_grenning__g (78).jpg"], ["P165", "archive1_train_grenning__g (14).jpg", "archive1_train_grenning__g (203).jpg"], ["P166", "archive1_test_blackspot__b (11).jpg", "archive1_train_blackspot__b (36).jpg"], ["P167", "archive1_train_blackspot__b (184).jpg", "archive1_train_blackspot__b (44).jpg"], ["P168", "archive1_train_grenning__g (131).jpg", "archive1_train_grenning__g (213).jpg"], ["P169", "archive1_train_blackspot__b (72).jpg", "archive1_train_blackspot__b (55).jpg"], ["P170", "archive1_test_blackspot__b (13).jpg", "archive1_train_blackspot__b (161).jpg"], ["P171", "archive1_train_blackspot__b (181).jpg", "archive1_train_blackspot__b (22).jpg"], ["P172", "archive1_train_grenning__g (97).jpg", "archive1_train_grenning__g (280).jpg"], ["P173", "archive1_train_blackspot__b (78).jpg", "archive1_train_blackspot__b (165).jpg"], ["P174", "archive1_train_canker__c (52).jpg", "archive1_train_canker__c (56).jpg"], ["P175", "archive1_train_grenning__g (172).jpg", "archive1_train_grenning__g (58).jpg"], ["P176", "archive1_train_grenning__g (160).jpg", "archive1_train_grenning__g (279).jpg"], ["P177", "archive1_test_canker__c (21).jpg", "archive1_train_canker__c (80).jpg"], ["P178", "archive1_test_blackspot__b (10).jpg", "archive1_train_blackspot__b (106).jpg"], ["P179", "archive1_train_grenning__g (243).jpg", "archive1_train_grenning__g (292).jpg"], ["P180", "archive1_train_blackspot__b (183).jpg", "archive1_train_blackspot__b (64).jpg"], ["P181", "archive1_train_grenning__g (312).jpg", "archive1_train_grenning__g (66).jpg"], ["P182", "archive1_train_blackspot__b (92).jpg", "archive1_train_blackspot__b (129).jpg"], ["P183", "archive1_train_canker__c (6).jpg", "archive1_train_canker__c (99).jpg"], ["P184", "archive1_train_grenning__g (156).jpg", "archive1_train_grenning__g (40).jpg"], ["P185", "archive1_train_grenning__g (18).jpg", "archive1_train_grenning__g (42).jpg"], ["P186", "archive1_train_blackspot__b (15).jpg", "archive1_train_blackspot__b (26).jpg"], ["P187", "archive1_train_blackspot__b (102).jpg", "archive1_train_blackspot__b (13).jpg"], ["P188", "archive1_train_canker__c (56).jpg", "archive1_train_canker__c (50).jpg"], ["P189", "archive1_train_blackspot__b (120).jpg", "archive1_test_blackspot__b (17).jpg"], ["P190", "archive1_train_grenning__g (121).jpg", "archive1_train_grenning__g (1).jpg"], ["P191", "archive1_train_blackspot__b (104).jpg", "archive1_train_blackspot__b (19).jpg"], ["P192", "archive1_train_grenning__g (275).jpg", "archive1_train_grenning__g (281).jpg"], ["P193", "archive1_train_grenning__g (174).jpg", "archive1_train_grenning__g (41).jpg"], ["P194", "archive1_train_blackspot__b (101).jpg", "archive1_train_blackspot__b (175).jpg"], ["P195", "archive1_train_grenning__g (100).jpg", "archive1_train_grenning__g (177).jpg"], ["P196", "archive1_train_blackspot__b (44).jpg", "archive1_train_blackspot__b (69).jpg"], ["P197", "archive1_train_grenning__g (256).jpg", "archive1_train_grenning__g (92).jpg"], ["P198", "citrus_Citrus_Leaves_greening__g (174).png", "citrus_Citrus_Leaves_greening__g (180).png"], ["P199", "archive1_train_grenning__g (285).jpg", "archive1_train_grenning__g (271).jpg"], ["P200", "archive1_train_grenning__g (306).jpg", "archive1_train_grenning__g (28).jpg"], ["P201", "archive1_train_grenning__g (106).jpg", "archive1_train_grenning__g (44).jpg"], ["P202", "archive1_train_canker__c (112).jpg", "archive1_train_canker__c (83).jpg"], ["P203", "archive1_test_canker__c (17).jpg", "archive1_train_canker__c (73).jpg"], ["P204", "archive1_train_blackspot__b (20).jpg", "archive1_train_blackspot__b (72).jpg"], ["P205", "archive1_train_grenning__g (221).jpg", "archive1_train_grenning__g (289).jpg"], ["P206", "archive1_train_grenning__g (285).jpg", "archive1_train_grenning__g (305).jpg"], ["P207", "archive1_train_grenning__g (232).jpg", "archive1_train_grenning__g (251).jpg"], ["P208", "archive1_train_grenning__g (299).jpg", "archive1_train_grenning__g (86).jpg"], ["P209", "archive1_train_grenning__g (303).jpg", "archive1_train_grenning__g (210).jpg"], ["P210", "archive1_train_grenning__g (166).jpg", "archive1_train_grenning__g (281).jpg"], ["P211", "archive1_test_blackspot__b (8).jpg", "archive1_train_blackspot__b (134).jpg"], ["P212", "archive1_train_grenning__g (137).jpg", "archive1_train_grenning__g (280).jpg"], ["P213", "archive1_train_blackspot__b (136).jpg", "archive1_train_blackspot__b (106).jpg"], ["P214", "archive1_train_grenning__g (192).jpg", "archive1_train_grenning__g (27).jpg"], ["P215", "archive1_train_blackspot__b (84).jpg", "archive1_train_blackspot__b (55).jpg"], ["P216", "archive1_test_blackspot__b (9).jpg", "archive1_train_blackspot__b (58).jpg"], ["P217", "archive1_train_grenning__g (286).jpg", "archive1_train_grenning__g (221).jpg"], ["P218", "archive1_train_canker__c (57).jpg", "archive1_train_canker__c (126).jpg"], ["P219", "archive1_test_grenning__g (13).jpg", "archive1_train_grenning__g (331).jpg"], ["P220", "archive1_train_grenning__g (279).jpg", "archive1_train_grenning__g (175).jpg"], ["P221", "citrus_Citrus_Leaves_canker__148.p.png", "citrus_Citrus_Leaves_canker__147.p.png"], ["P222", "archive1_train_grenning__g (135).jpg", "archive1_train_grenning__g (178).jpg"], ["P223", "archive1_train_grenning__g (6).jpg", "archive1_train_grenning__g (214).jpg"], ["P224", "archive1_train_blackspot__b (34).jpg", "archive1_train_blackspot__b (86).jpg"], ["P225", "archive1_train_canker__c (166).jpg", "archive1_train_canker__c (55).jpg"], ["P226", "archive1_train_grenning__g (165).jpg", "archive1_train_grenning__g (72).jpg"], ["P227", "archive1_train_grenning__g (240).jpg", "archive1_train_grenning__g (335).jpg"], ["P228", "archive1_train_grenning__g (21).jpg", "archive1_train_grenning__g (36).jpg"], ["P229", "archive1_train_grenning__g (39).jpg", "archive1_train_grenning__g (41).jpg"], ["P230", "archive1_train_blackspot__b (104).jpg", "archive1_train_blackspot__b (39).jpg"], ["P231", "archive1_train_blackspot__b (38).jpg", "archive1_train_blackspot__b (39).jpg"], ["P232", "archive1_train_blackspot__b (20).jpg", "archive1_train_blackspot__b (71).jpg"], ["P233", "archive1_train_blackspot__b (44).jpg", "archive1_train_blackspot__b (35).jpg"], ["P234", "archive1_train_grenning__g (22).jpg", "archive1_train_grenning__g (157).jpg"], ["P235", "archive1_train_blackspot__b (174).jpg", "archive1_train_blackspot__b (68).jpg"], ["P236", "archive1_train_blackspot__b (36).jpg", "archive1_train_blackspot__b (8).jpg"], ["P237", "archive1_train_grenning__g (167).jpg", "archive1_train_grenning__g (15).jpg"], ["P238", "archive1_train_grenning__g (306).jpg", "archive1_train_grenning__g (320).jpg"], ["P239", "archive1_test_grenning__g (4).jpg", "archive1_train_grenning__g (153).jpg"], ["P240", "archive1_train_blackspot__b (173).jpg", "archive1_train_blackspot__b (55).jpg"], ["P241", "archive1_train_blackspot__b (38).jpg", "archive1_train_blackspot__b (71).jpg"], ["P242", "archive1_test_grenning__g (19).jpg", "archive1_train_grenning__g (186).jpg"], ["P243", "archive1_train_grenning__g (179).jpg", "archive1_train_grenning__g (99).jpg"], ["P244", "archive1_train_grenning__g (21).jpg", "archive1_train_grenning__g (58).jpg"], ["P245", "archive1_train_grenning__g (191).jpg", "archive1_train_grenning__g (275).jpg"], ["P246", "archive1_train_canker__c (52).jpg", "archive1_train_canker__c (27).jpg"], ["P247", "archive1_train_blackspot__b (184).jpg", "archive1_train_blackspot__b (183).jpg"], ["P248", "archive1_train_grenning__g (35).jpg", "archive1_train_grenning__g (152).jpg"], ["P249", "archive1_train_blackspot__b (140).jpg", "archive1_train_blackspot__b (177).jpg"], ["P250", "archive1_train_blackspot__b (44).jpg", "archive1_train_blackspot__b (52).jpg"], ["P251", "archive1_train_grenning__g (122).jpg", "archive1_train_grenning__g (239).jpg"], ["P252", "archive1_train_canker__c (52).jpg", "archive1_train_canker__c (109).jpg"], ["P253", "archive1_train_grenning__g (107).jpg", "archive1_train_grenning__g (39).jpg"], ["P254", "archive1_train_grenning__g (60).jpg", "archive1_train_grenning__g (92).jpg"], ["P255", "archive1_train_grenning__g (44).jpg", "archive1_train_grenning__g (298).jpg"], ["P256", "archive1_train_grenning__g (206).jpg", "archive1_train_grenning__g (36).jpg"], ["P257", "archive1_test_grenning__g (12).jpg", "archive1_train_grenning__g (194).jpg"], ["P258", "archive1_train_grenning__g (323).jpg", "archive1_train_grenning__g (63).jpg"], ["P259", "archive1_train_grenning__g (117).jpg", "archive1_train_grenning__g (5).jpg"], ["P260", "archive1_train_grenning__g (21).jpg", "archive1_train_grenning__g (25).jpg"], ["P261", "archive1_train_grenning__g (120).jpg", "archive1_train_grenning__g (334).jpg"], ["P262", "archive1_train_grenning__g (337).jpg", "archive1_train_grenning__g (267).jpg"], ["P263", "archive1_test_blackspot__b (14).jpg", "archive1_train_blackspot__b (71).jpg"], ["P264", "archive1_train_blackspot__b (16).jpg", "archive1_train_blackspot__b (45).jpg"], ["P265", "archive1_train_blackspot__b (113).jpg", "archive1_train_blackspot__b (72).jpg"], ["P266", "archive1_train_blackspot__b (37).jpg", "archive1_train_blackspot__b (141).jpg"], ["P267", "archive1_train_blackspot__b (112).jpg", "archive1_train_blackspot__b (92).jpg"], ["P268", "archive1_train_blackspot__b (31).jpg", "archive1_train_blackspot__b (33).jpg"], ["P269", "archive1_train_grenning__g (10).jpg", "archive1_train_grenning__g (241).jpg"], ["P270", "archive1_train_blackspot__b (31).jpg", "archive1_train_blackspot__b (56).jpg"], ["P271", "archive1_train_grenning__g (152).jpg", "archive1_train_grenning__g (271).jpg"], ["P272", "archive1_train_blackspot__b (174).jpg", "archive1_train_blackspot__b (38).jpg"], ["P273", "archive1_train_blackspot__b (178).jpg", "archive1_train_blackspot__b (179).jpg"], ["P274", "archive1_train_grenning__g (205).jpg", "archive1_train_grenning__g (58).jpg"], ["P275", "archive1_train_grenning__g (118).jpg", "archive1_train_grenning__g (86).jpg"], ["P276", "archive1_train_grenning__g (307).jpg", "archive1_train_grenning__g (72).jpg"], ["P277", "archive1_train_grenning__g (170).jpg", "archive1_train_grenning__g (81).jpg"], ["P278", "archive1_train_grenning__g (120).jpg", "archive1_train_grenning__g (103).jpg"], ["P279", "archive1_train_blackspot__b (15).jpg", "archive1_train_blackspot__b (71).jpg"], ["P280", "archive1_train_grenning__g (285).jpg", "archive1_train_grenning__g (214).jpg"], ["P281", "archive1_train_blackspot__b (115).jpg", "archive1_train_blackspot__b (124).jpg"], ["P282", "archive1_train_grenning__g (131).jpg", "archive1_train_grenning__g (21).jpg"], ["P283", "archive1_train_grenning__g (259).jpg", "archive1_train_grenning__g (303).jpg"], ["P284", "archive1_train_grenning__g (120).jpg", "archive1_train_grenning__g (283).jpg"], ["P285", "archive1_train_grenning__g (205).jpg", "archive1_train_grenning__g (332).jpg"], ["P286", "archive1_train_blackspot__b (154).jpg", "archive1_train_blackspot__b (32).jpg"], ["P287", "archive1_test_blackspot__b (13).jpg", "archive1_train_blackspot__b (54).jpg"], ["P288", "archive1_train_grenning__g (261).jpg", "archive1_train_grenning__g (310).jpg"], ["P289", "archive1_train_blackspot__b (16).jpg", "archive1_train_blackspot__b (66).jpg"], ["P290", "archive1_train_grenning__g (307).jpg", "archive1_train_grenning__g (77).jpg"], ["P291", "archive1_train_grenning__g (117).jpg", "archive1_train_grenning__g (52).jpg"], ["P292", "archive1_train_blackspot__b (110).jpg", "archive1_train_blackspot__b (160).jpg"], ["P293", "archive1_test_grenning__g (1).jpg", "archive1_train_grenning__g (78).jpg"], ["P294", "archive1_train_blackspot__b (115).jpg", "archive1_train_blackspot__b (133).jpg"], ["P295", "archive1_train_fresh__f (168).png", "archive1_train_fresh__f (269).png"], ["P296", "archive1_train_grenning__g (6).jpg", "archive1_train_grenning__g (271).jpg"], ["P297", "archive1_test_blackspot__b (10).jpg", "archive1_train_blackspot__b (44).jpg"], ["P298", "archive1_train_fresh__f (209).png", "archive1_train_fresh__f (241).png"], ["P299", "archive1_test_blackspot__b (4).jpg", "archive1_train_blackspot__b (37).jpg"], ["P300", "archive1_train_grenning__g (19).jpg", "archive1_train_grenning__g (192).jpg"], ["P301", "archive1_train_canker__c (167).jpg", "archive1_train_canker__c (172).jpg"], ["P302", "archive1_train_blackspot__b (104).jpg", "archive1_train_blackspot__b (72).jpg"], ["P303", "archive1_train_grenning__g (15).jpg", "archive1_train_grenning__g (83).jpg"], ["P304", "archive1_train_grenning__g (320).jpg", "archive1_train_grenning__g (26).jpg"], ["P305", "archive1_train_grenning__g (279).jpg", "archive1_train_grenning__g (203).jpg"], ["P306", "archive1_train_grenning__g (137).jpg", "archive1_train_grenning__g (167).jpg"], ["P307", "archive1_train_blackspot__b (183).jpg", "archive1_train_blackspot__b (69).jpg"], ["P308", "archive1_train_grenning__g (173).jpg", "archive1_train_grenning__g (58).jpg"], ["P309", "archive1_train_blackspot__b (128).jpg", "archive1_train_blackspot__b (96).jpg"], ["P310", "archive1_test_blackspot__b (16).jpg", "archive1_test_blackspot__b (19).jpg"], ["P311", "archive1_train_grenning__g (23).jpg", "archive1_train_grenning__g (297).jpg"], ["P312", "archive1_test_grenning__g (3).jpg", "archive1_train_grenning__g (269).jpg"], ["P313", "archive1_train_canker__c (52).jpg", "archive1_train_canker__c (85).jpg"], ["P314", "archive1_train_canker__c (146).jpg", "archive1_train_canker__c (51).jpg"], ["P315", "archive1_train_grenning__g (281).jpg", "archive1_train_grenning__g (337).jpg"], ["P316", "archive1_train_blackspot__b (104).jpg", "archive1_train_blackspot__b (176).jpg"], ["P317", "archive1_train_canker__c (163).jpg", "archive1_train_canker__c (178).jpg"], ["P318", "archive1_train_grenning__g (108).jpg", "archive1_train_grenning__g (23).jpg"], ["P319", "archive1_train_blackspot__b (16).jpg", "archive1_train_blackspot__b (65).jpg"], ["P320", "archive1_train_blackspot__b (23).jpg", "archive1_train_blackspot__b (48).jpg"], ["P321", "archive1_train_blackspot__b (153).jpg", "archive1_train_blackspot__b (4).jpg"], ["P322", "archive1_train_blackspot__b (20).jpg", "archive1_train_blackspot__b (46).jpg"], ["P323", "archive1_train_blackspot__b (181).jpg", "archive1_train_blackspot__b (32).jpg"], ["P324", "archive1_train_canker__c (35).jpg", "archive1_train_canker__c (51).jpg"], ["P325", "archive1_train_blackspot__b (38).jpg", "archive1_train_blackspot__b (65).jpg"], ["P326", "archive1_train_grenning__g (191).jpg", "archive1_train_grenning__g (207).jpg"], ["P327", "archive1_train_grenning__g (211).jpg", "archive1_train_grenning__g (293).jpg"], ["P328", "archive1_train_grenning__g (168).jpg", "archive1_train_grenning__g (62).jpg"], ["P329", "archive1_train_blackspot__b (181).jpg", "archive1_train_blackspot__b (35).jpg"], ["P330", "archive1_train_grenning__g (25).jpg", "archive1_train_grenning__g (28).jpg"], ["P331", "archive1_train_grenning__g (116).jpg", "archive1_train_grenning__g (293).jpg"], ["P332", "archive1_train_grenning__g (39).jpg", "archive1_train_grenning__g (95).jpg"], ["P333", "archive1_train_canker__c (26).jpg", "archive1_train_canker__c (144).jpg"], ["P334", "archive1_train_grenning__g (307).jpg", "archive1_train_grenning__g (291).jpg"], ["P335", "archive1_train_grenning__g (75).jpg", "archive1_train_grenning__g (335).jpg"], ["P336", "archive1_test_blackspot__b (20).jpg", "archive1_train_blackspot__b (60).jpg"], ["P337", "archive1_train_canker__c (56).jpg", "archive1_train_canker__c (74).jpg"], ["P338", "archive1_train_canker__c (52).jpg", "archive1_train_canker__c (128).jpg"], ["P339", "archive1_train_grenning__g (39).jpg", "archive1_train_grenning__g (220).jpg"], ["P340", "archive1_train_grenning__g (205).jpg", "archive1_train_grenning__g (21).jpg"], ["P341", "archive1_train_grenning__g (75).jpg", "archive1_train_grenning__g (58).jpg"], ["P342", "archive1_train_canker__c (171).jpg", "archive1_train_canker__c (75).jpg"], ["P343", "archive1_train_blackspot__b (174).jpg", "archive1_train_blackspot__b (46).jpg"], ["P344", "archive1_train_grenning__g (139).jpg", "archive1_train_grenning__g (215).jpg"], ["P345", "archive1_train_grenning__g (307).jpg", "archive1_train_grenning__g (52).jpg"], ["P346", "archive1_train_grenning__g (219).jpg", "archive1_train_grenning__g (209).jpg"], ["P347", "archive1_train_grenning__g (292).jpg", "archive1_train_grenning__g (296).jpg"], ["P348", "archive1_train_grenning__g (6).jpg", "archive1_train_grenning__g (215).jpg"], ["P349", "archive1_test_blackspot__b (1).jpg", "archive1_test_blackspot__b (18).jpg"], ["P350", "archive1_train_grenning__g (272).jpg", "archive1_train_grenning__g (163).jpg"], ["P351", "archive1_train_blackspot__b (134).jpg", "archive1_train_blackspot__b (135).jpg"], ["P352", "archive1_train_canker__c (154).jpg", "archive1_train_canker__c (126).jpg"], ["P353", "archive1_test_blackspot__b (13).jpg", "archive1_train_blackspot__b (175).jpg"], ["P354", "archive1_test_canker__c (14).jpg", "archive1_train_canker__c (29).jpg"], ["P355", "archive1_train_grenning__g (213).jpg", "archive1_train_grenning__g (37).jpg"], ["P356", "archive1_train_grenning__g (116).jpg", "archive1_train_grenning__g (276).jpg"], ["P357", "archive1_train_blackspot__b (111).jpg", "archive1_train_blackspot__b (98).jpg"], ["P358", "archive1_train_grenning__g (139).jpg", "archive1_train_grenning__g (280).jpg"], ["P359", "archive1_train_blackspot__b (8).jpg", "archive1_train_blackspot__b (121).jpg"], ["P360", "archive1_train_canker__c (19).jpg", "archive1_train_canker__c (17).jpg"], ["P361", "archive1_train_blackspot__b (44).jpg", "archive1_train_blackspot__b (19).jpg"], ["P362", "archive1_train_blackspot__b (120).jpg", "archive1_train_blackspot__b (178).jpg"], ["P363", "archive1_train_blackspot__b (45).jpg", "archive1_train_blackspot__b (46).jpg"], ["P364", "archive1_train_grenning__g (137).jpg", "archive1_train_grenning__g (337).jpg"], ["P365", "archive1_train_canker__c (177).jpg", "archive1_train_canker__c (92).jpg"], ["P366", "archive1_train_blackspot__b (3).jpg", "archive1_train_blackspot__b (73).jpg"], ["P367", "archive1_train_blackspot__b (105).jpg", "archive1_train_blackspot__b (151).jpg"], ["P368", "archive1_train_grenning__g (165).jpg", "archive1_train_grenning__g (35).jpg"], ["P369", "archive1_train_grenning__g (130).jpg", "archive1_train_grenning__g (321).jpg"], ["P370", "archive1_test_blackspot__b (12).jpg", "archive1_train_blackspot__b (168).jpg"], ["P371", "archive1_train_grenning__g (23).jpg", "archive1_train_grenning__g (246).jpg"], ["P372", "archive1_train_blackspot__b (139).jpg", "archive1_train_blackspot__b (68).jpg"], ["P373", "archive1_test_canker__c (9).jpg", "archive1_train_canker__c (13).jpg"], ["P374", "archive1_train_blackspot__b (181).jpg", "archive1_train_blackspot__b (11).jpg"], ["P375", "archive1_train_canker__c (83).jpg", "archive1_train_canker__c (24).jpg"], ["P376", "archive1_train_blackspot__b (173).jpg", "archive1_train_blackspot__b (29).jpg"], ["P377", "archive1_train_blackspot__b (104).jpg", "archive1_train_blackspot__b (8).jpg"], ["P378", "archive1_train_grenning__g (121).jpg", "archive1_train_grenning__g (180).jpg"], ["P379", "archive1_train_grenning__g (160).jpg", "archive1_train_grenning__g (12).jpg"], ["P380", "archive1_train_canker__c (48).jpg", "archive1_train_canker__c (85).jpg"], ["P381", "archive1_train_grenning__g (187).jpg", "archive1_train_grenning__g (295).jpg"], ["P382", "archive1_train_grenning__g (278).jpg", "archive1_train_grenning__g (267).jpg"], ["P383", "archive1_train_grenning__g (167).jpg", "archive1_train_grenning__g (347).jpg"], ["P384", "archive1_train_blackspot__b (134).jpg", "archive1_train_blackspot__b (44).jpg"], ["P385", "archive1_test_canker__c (20).jpg", "archive1_train_canker__c (100).jpg"], ["P386", "archive1_train_blackspot__b (22).jpg", "archive1_train_blackspot__b (38).jpg"], ["P387", "archive1_train_grenning__g (10).jpg", "archive1_train_grenning__g (186).jpg"], ["P388", "archive1_train_blackspot__b (79).jpg", "archive1_train_blackspot__b (145).jpg"], ["P389", "archive1_train_canker__c (145).jpg", "archive1_train_canker__c (55).jpg"], ["P390", "archive1_train_grenning__g (330).jpg", "archive1_train_grenning__g (209).jpg"], ["P391", "archive1_train_grenning__g (191).jpg", "archive1_train_grenning__g (11).jpg"], ["P392", "archive1_test_grenning__g (1).jpg", "archive1_train_grenning__g (205).jpg"], ["P393", "archive1_train_grenning__g (162).jpg", "archive1_train_grenning__g (17).jpg"], ["P394", "archive1_train_canker__c (155).jpg", "archive1_train_canker__c (29).jpg"], ["P395", "archive1_test_grenning__g (13).jpg", "archive1_train_grenning__g (214).jpg"], ["P396", "archive1_train_grenning__g (276).jpg", "archive1_train_grenning__g (80).jpg"], ["P397", "archive1_train_grenning__g (136).jpg", "archive1_train_grenning__g (35).jpg"], ["P398", "archive1_train_blackspot__b (31).jpg", "archive1_train_blackspot__b (4).jpg"], ["P399", "archive1_train_grenning__g (133).jpg", "archive1_train_grenning__g (43).jpg"], ["P400", "archive1_train_blackspot__b (178).jpg", "archive1_test_blackspot__b (17).jpg"]]"""
#@title Cell 6 — Quick tasks: EXIF (M14), fruit-only source probe (3.1), pair sheets (C4, C5)
import json as _json
from PIL import ExifTags, ImageDraw, ImageFont
# ---- M14: EXIF timestamps
ex = []
for p, f in zip(df.path, df.file):
    try:
        e = Image.open(p).getexif(); sub = e.get_ifd(0x8769) if hasattr(e, "get_ifd") else {}
        ex.append(dict(file=f, datetime=e.get(306), datetime_original=sub.get(36867) if sub else None, camera=e.get(272), make=e.get(271), width=Image.open(p).size[0], height=Image.open(p).size[1]))
    except Exception as err:
        ex.append(dict(file=f, datetime=None, datetime_original=None, camera=None, make=None, width=None, height=None))
EX = pd.DataFrame(ex); EX.to_csv(f"{OUT}/tables/exif.csv", index=False)
print("Images with any EXIF timestamp:", int(EX[["datetime", "datetime_original"]].notna().any(axis=1).sum()), "of", len(EX))
# ---- 3.1: source probe restricted to fruit images without black corners
PROBE_CSV = f"{OUT}/tables/probe_fruit_nonrotated.csv"
if not os.path.exists(PROBE_CSV):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict
    from sklearn.metrics import roc_auc_score
    fe = keras.applications.MobileNetV2(include_top=False, pooling="avg", weights="imagenet")
    def feats(Xa):
        return np.concatenate([fe.predict(keras.applications.mobilenet_v2.preprocess_input(Xa[s:s + 128].astype("float32")), verbose=0) for s in range(0, len(Xa), 128)])
    sel = np.where((df.organ.values == "fruit") & (~df.pre_rotated.values))[0]
    src = (df.source.values[sel] == "Silva").astype(int)
    rows_p = []
    for nm, Xs in [("full image", X[sel]), ("corner patches only", np.stack([corners_only(x) for x in X[sel]]))]:
        F = feats(Xs)
        k = int(min(5, pd.Series(G[sel]).groupby(src).nunique().min()))
        pr_ = cross_val_predict(LogisticRegression(max_iter=3000), F, src, groups=G[sel], cv=StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
        rows_p.append(dict(subset="fruit, no black corners", input=nm, n=len(sel), n_sargodha=int((src == 0).sum()), n_orange=int((src == 1).sum()),
                           acc=float(((pr_ > .5) == src).mean()), auc=float(roc_auc_score(src, pr_)), majority=float(max(src.mean(), 1 - src.mean()))))
        for c in ["Black_Spot", "Canker", "Greening", "Healthy"]:
            ii = np.where(df.cls.values[sel] == c)[0]
            if len(np.unique(src[ii])) == 2 and min(np.bincount(src[ii])) >= 3:
                kk = int(min(3, pd.Series(G[sel][ii]).groupby(src[ii]).nunique().min()))
                if kk >= 2:
                    pc_ = cross_val_predict(LogisticRegression(max_iter=3000), F[ii], src[ii], groups=G[sel][ii], cv=StratifiedGroupKFold(n_splits=kk, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
                    rows_p.append(dict(subset=f"fruit, no black corners, {c}", input=nm, n=len(ii), n_sargodha=int((src[ii] == 0).sum()), n_orange=int((src[ii] == 1).sum()), auc=float(roc_auc_score(src[ii], pc_))))
    pd.DataFrame(rows_p).to_csv(PROBE_CSV, index=False); del fe; free_memory()
display(pd.read_csv(PROBE_CSV).round(3))
# ---- C4 / C5: image sheets (10 pairs per sheet)
C5_PAIRS = _json.loads(C5_JSON)
C4_DF = pd.read_csv(f"{OUT}/tables/prev_cross_class_near_duplicates.csv")
C4_PAIRS = [[f"X{i + 1:03d}", a, b] for i, (a, b) in enumerate(zip(C4_DF.file_a, C4_DF.file_b))]
pos = {f: i for i, f in enumerate(df.file)}
def sheet(pairs, prefix):
    T = 230; per = 10
    for s in range(0, len(pairs), per):
        chunk = pairs[s:s + per]
        canvas = Image.new("RGB", (2 * (2 * T + 30) + 20, 5 * (T + 34) + 10), "white"); d = ImageDraw.Draw(canvas)
        for q, (pid, fa, fb) in enumerate(chunk):
            col, row = q % 2, q // 2; x0 = 10 + col * (2 * T + 30); y0 = 10 + row * (T + 34)
            d.text((x0, y0), pid, fill="black")
            for t, f in enumerate([fa, fb]):
                canvas.paste(Image.fromarray(X[pos[f]]).resize((T, T)), (x0 + t * (T + 6), y0 + 14))
        canvas.save(f"{OUT}/sheets/{prefix}_{s // per + 1:02d}.jpg", quality=82)
if not os.path.exists(f"{OUT}/sheets/C5_01.jpg"):
    sheet(C4_PAIRS, "C4"); sheet(C5_PAIRS, "C5")
print("sheets:", len(os.listdir(f"{OUT}/sheets")))


In [ ]:
#@title Cell 7 — C3: deduplicated arm (one random image per group, ordinary stratified 5-fold CV)
from sklearn.model_selection import StratifiedKFold
rng = np.random.default_rng(SEED)
rep = np.array(sorted(rng.choice(np.where(G == g)[0]) for g in np.unique(G)))
pd.DataFrame({"file": df.file.values[rep], "cls": df.cls.values[rep], "group": G[rep]}).to_csv(f"{OUT}/tables/dedup_representatives.csv", index=False)
print("representatives:", len(rep), dict(pd.Series(df.cls.values[rep]).value_counts()))
DFOLD = np.full(len(rep), -1)
for k, (_, te) in enumerate(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(rep)), Y[rep])):
    DFOLD[te] = k
LOG_D = f"{OUT}/tables/dedup_cv_log.csv"
for name in TWO:
    for k in range(N_FOLDS):
        L = pd.read_csv(LOG_D) if os.path.exists(LOG_D) else pd.DataFrame(columns=["model", "fold"])
        if ((L.model == name) & (L.fold == k)).any(): print("skip:", name, k); continue
        t0 = time.time()
        te = rep[DFOLD == k]; rest = rep[DFOLD != k]
        tr_i, va_i = next(StratifiedKFold(n_splits=6, shuffle=True, random_state=SEED + k).split(np.zeros(len(rest)), Y[rest]))
        tr, va = rest[tr_i], rest[va_i]
        model, base = train_two(name, X[tr], Y[tr], X[va], Y[va], NC, Mtr=None, seed=SEED)
        p = predict_u8(model, name, X[te]); pr = p.argmax(1)
        np.savez_compressed(f"{OUT}/oof_dedup/{name}_fold{k}.npz", te=te, p_full=p)
        append_log(LOG_D, dict(model=name, fold=k, n_test=len(te), acc=accuracy_score(Y[te], pr), macro_f1=mf1(Y[te], pr), minutes=(time.time() - t0) / 60))
        print(name, k, round(accuracy_score(Y[te], pr), 3)); del model, base; free_memory()
display(pd.read_csv(LOG_D).groupby("model")[["acc", "macro_f1"]].mean().round(3))


In [ ]:
#@title Cell 8 — M1/M2: leakage simulation on the Sargodha leaves, 5 seeds, protocols A, A′ and B
from sklearn.model_selection import train_test_split
def aug_offline(x, rng):
    im = Image.fromarray(x).rotate(90 * int(rng.integers(0, 4)))
    if rng.random() < .5: im = im.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < .5: im = im.transpose(Image.FLIP_TOP_BOTTOM)
    return np.asarray(im.rotate(float(rng.uniform(-30, 30))))
def balance(idx, target, rng):
    xs, ys, src, aug = [], [], [], []
    for c in np.unique(Y[idx]):
        ic = idx[Y[idx] == c]
        xs += [X[i] for i in ic]; ys += [c] * len(ic); src += list(ic); aug += [False] * len(ic)
        for j in range(max(0, target - len(ic))):
            i = ic[j % len(ic)]; xs.append(aug_offline(X[i], rng)); ys.append(c); src.append(i); aug.append(True)
    return np.stack(xs), np.array(ys, dtype="int64"), np.array(src), np.array(aug)
leaf = np.where((df.source.values == "Rauf") & (df.organ.values == "leaf"))[0]
TARGET = 204
LEAK_LOG = f"{OUT}/tables/leakage_simulation_5seeds.csv"
for name in TWO:
    for s in LEAK_SEEDS:
        L = pd.read_csv(LEAK_LOG) if os.path.exists(LEAK_LOG) else pd.DataFrame(columns=["protocol", "model", "seed"])
        rng = np.random.default_rng(s)
        if not ((L.protocol == "A") & (L.model == name) & (L.seed == s)).any():
            Xa, ya, sa, ia = balance(leaf, TARGET, rng)
            tr, te = train_test_split(np.arange(len(ya)), test_size=0.3, stratify=ya, random_state=s)
            tr, va = train_test_split(tr, test_size=0.15, stratify=ya[tr], random_state=s)
            m, _ = train_one(name, Xa[tr], ya[tr], Xa[va], ya[va], NC)
            p = predict_u8(m, name, Xa[te]); pr = p.argmax(1); free_memory()
            np.savez_compressed(f"{OUT}/oof_leak/A_{name}_seed{s}.npz", te=te, src=sa[te], is_aug=ia[te], y=ya[te], p=p)
            orig = ~ia[te]; shares = np.isin(sa[te], sa[tr])
            rows_ = [dict(protocol="A", model=name, seed=s, n_test=len(te), acc=accuracy_score(ya[te], pr), macro_f1=mf1(ya[te], pr), test_shares_original_with_train=float(shares.mean())),
                     dict(protocol="A_prime", model=name, seed=s, n_test=int(orig.sum()), acc=accuracy_score(ya[te][orig], pr[orig]), macro_f1=mf1(ya[te][orig], pr[orig]),
                          test_shares_original_with_train=float(shares[orig].mean()))]
            for r_ in rows_: append_log(LEAK_LOG, r_)
            print(rows_)
        L = pd.read_csv(LEAK_LOG)
        if not ((L.protocol == "B") & (L.model == name) & (L.seed == s)).any():
            g = G[leaf]
            o_tr, o_te = next(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=s).split(np.zeros(len(leaf)), Y[leaf], g))
            o_tr, o_te = leaf[o_tr], leaf[o_te]
            i_tr, i_va = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s).split(np.zeros(len(o_tr)), Y[o_tr], G[o_tr]))
            Xb, yb, _, _ = balance(o_tr[i_tr], int(TARGET * 0.7), rng)
            m, _ = train_one(name, Xb, yb, X[o_tr[i_va]], Y[o_tr[i_va]], NC)
            p = predict_u8(m, name, X[o_te]); pr = p.argmax(1); free_memory()
            np.savez_compressed(f"{OUT}/oof_leak/B_{name}_seed{s}.npz", te=o_te, y=Y[o_te], p=p)
            r_ = dict(protocol="B", model=name, seed=s, n_test=len(o_te), acc=accuracy_score(Y[o_te], pr), macro_f1=mf1(Y[o_te], pr), test_shares_original_with_train=0.0)
            append_log(LEAK_LOG, r_); print(r_)
display(pd.read_csv(LEAK_LOG).groupby(["protocol", "model"])[["acc", "macro_f1", "test_shares_original_with_train"]].agg(["mean", "std"]).round(3))


In [ ]:
#@title Cell 9 — 2.10: grouped 5-fold CV within each dataset (removes class–source confounding)
LOG_W = f"{OUT}/tables/within_dataset_cv_log.csv"
for dsname, srcv in [("Sargodha", "Rauf"), ("OrangeDiseases", "Silva")]:
    ii = np.where(df.source.values == srcv)[0]
    present = sorted(np.unique(Y[ii])); ymap = {c: j for j, c in enumerate(present)}; yy = np.array([ymap[c] for c in Y[ii]])
    WF = np.full(len(ii), -1)
    for k, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(ii)), yy, G[ii])):
        WF[te] = k
    for name in TWO:
        for k in range(N_FOLDS):
            L = pd.read_csv(LOG_W) if os.path.exists(LOG_W) else pd.DataFrame(columns=["dataset", "model", "fold"])
            if ((L.dataset == dsname) & (L.model == name) & (L.fold == k)).any(): print("skip:", dsname, name, k); continue
            t0 = time.time()
            te = np.where(WF == k)[0]; rest = np.where(WF != k)[0]
            tr_i, va_i = next(StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED + k).split(np.zeros(len(rest)), yy[rest], G[ii][rest]))
            tr, va = rest[tr_i], rest[va_i]
            assert not (set(G[ii][tr]) & set(G[ii][te])) and not (set(G[ii][va]) & set(G[ii][te]))
            model, base = train_two(name, X[ii][tr], yy[tr], X[ii][va], yy[va], len(present), Mtr=None, seed=SEED)
            p = predict_u8(model, name, X[ii][te]); pr = p.argmax(1)
            np.savez_compressed(f"{OUT}/oof_within/{dsname}_{name}_fold{k}.npz", te=ii[te], y=yy[te], p_full=p, classes=np.array(present))
            append_log(LOG_W, dict(dataset=dsname, model=name, fold=k, n_test=len(te), acc=accuracy_score(yy[te], pr), macro_f1=mf1(yy[te], pr), minutes=(time.time() - t0) / 60))
            print(dsname, name, k, round(accuracy_score(yy[te], pr), 3)); del model, base; free_memory()
display(pd.read_csv(LOG_W).groupby(["dataset", "model"])[["acc", "macro_f1"]].mean().round(3))


In [ ]:
#@title Cell 10 — M3: random (ungrouped) stratified 5-fold CV for the other five architectures (same folds as the v4 run)
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
RFOLD = np.full(N, -1)
for k, (_, te) in enumerate(skf.split(np.zeros(N), Y)):
    RFOLD[te] = k
def split_random(k):
    te = np.where(RFOLD == k)[0]; rest = np.where(RFOLD != k)[0]
    inner = StratifiedKFold(n_splits=6, shuffle=True, random_state=SEED + k)
    tr_i, va_i = next(inner.split(np.zeros(len(rest)), Y[rest]))
    return rest[tr_i], rest[va_i], te
LOG_R = f"{OUT}/tables/random_cv_log_other5.csv"
for name in [m for m in ALL_MODELS if m not in TWO]:
    for k in range(N_FOLDS):
        L = pd.read_csv(LOG_R) if os.path.exists(LOG_R) else pd.DataFrame(columns=["model", "fold"])
        if ((L.model == name) & (L.fold == k)).any(): print("skip:", name, k); continue
        t0 = time.time(); tr, va, te = split_random(k)
        leaky = np.isin(G[te], G[np.concatenate([tr, va])])
        model, base = train_two(name, X[tr], Y[tr], X[va], Y[va], NC, Mtr=None, seed=SEED)
        p = predict_u8(model, name, X[te]); pr = p.argmax(1)
        np.savez_compressed(f"{OUT}/oof_random/{name}_fold{k}.npz", te=te, p_full=p, leaky=leaky)
        append_log(LOG_R, dict(model=name, fold=k, n_test=len(te), acc=accuracy_score(Y[te], pr), macro_f1=mf1(Y[te], pr), frac_leaky=float(leaky.mean()), minutes=(time.time() - t0) / 60))
        print(name, k, round(accuracy_score(Y[te], pr), 3)); del model, base; free_memory()
display(pd.read_csv(LOG_R).groupby("model")[["acc", "macro_f1"]].mean().round(3))


In [ ]:
#@title Cell 11 — M4: grouped 5-fold CV with two further fold assignments (seeds 0 and 1), all seven architectures
LOG_M = f"{OUT}/tables/repeated_grouped_cv_log.csv"
for fs in REPEAT_FOLD_SEEDS:
    RF = np.full(N, -1)
    for k, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=fs).split(np.zeros(N), Y, G)):
        RF[te] = k
    pd.DataFrame({"file": df.file, "fold": RF}).to_csv(f"{OUT}/tables/folds_seed{fs}.csv", index=False)
    for name in ALL_MODELS:
        for k in range(N_FOLDS):
            L = pd.read_csv(LOG_M) if os.path.exists(LOG_M) else pd.DataFrame(columns=["fold_seed", "model", "fold"])
            if ((L.fold_seed == fs) & (L.model == name) & (L.fold == k)).any(): print("skip:", fs, name, k); continue
            t0 = time.time()
            te = np.where(RF == k)[0]; rest = np.where(RF != k)[0]
            tr_i, va_i = next(StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=fs * 100 + k).split(np.zeros(len(rest)), Y[rest], G[rest]))
            tr, va = rest[tr_i], rest[va_i]
            assert not (set(G[tr]) & set(G[te])) and not (set(G[va]) & set(G[te])) and not (set(G[tr]) & set(G[va]))
            model, base = train_two(name, X[tr], Y[tr], X[va], Y[va], NC, Mtr=None, seed=SEED)
            p = predict_u8(model, name, X[te]); pr = p.argmax(1)
            np.savez_compressed(f"{OUT}/oof_repeat/seed{fs}_{name}_fold{k}.npz", te=te, p_full=p)
            append_log(LOG_M, dict(fold_seed=fs, model=name, fold=k, n_test=len(te), acc=accuracy_score(Y[te], pr), macro_f1=mf1(Y[te], pr), minutes=(time.time() - t0) / 60))
            print(fs, name, k, round(accuracy_score(Y[te], pr), 3)); del model, base; free_memory()
L = pd.read_csv(LOG_M); L["correct"] = L.acc * L.n_test
print(L.groupby(["model", "fold_seed"]).apply(lambda d: d.correct.sum() / d.n_test.sum()).unstack().round(3))


In [ ]:
#@title Cell 12 — Pack results (download ONLY p1_part2_results.zip)
pack = "/kaggle/temp/pack_p1part2"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(OUT, pack)
z = shutil.make_archive("/kaggle/working/p1_part2_results", "zip", pack)
print("FILE TO DOWNLOAD:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
for f in sorted(glob.glob(f"{OUT}/tables/*.csv")): print(" ", os.path.basename(f))
